# FlashInfer-Bench Demo

**FlashInfer-Bench** is a benchmark suite and production workflow designed to build a virtuous cycle of self-improving AI systems. It enables AI agents and human experts to collaboratively optimize GPU kernels that power large language models.

- **Documentation**: [bench.flashinfer.ai/docs](https://bench.flashinfer.ai/docs/)
- **Leaderboard**: [bench.flashinfer.ai](https://bench.flashinfer.ai/)
- **Blog Post**: [flashinfer.ai/2025/10/21/flashinfer-bench](https://flashinfer.ai/2025/10/21/flashinfer-bench.html)
- **GitHub**: [github.com/flashinfer-ai/flashinfer-bench](https://github.com/flashinfer-ai/flashinfer-bench)


---
## 1. The FlashInfer-Bench Pipeline

FlashInfer-Bench creates a **virtuous cycle** where AI agents and human experts collaboratively optimize GPU kernels:

- **Definition** → Kernel contract (axes, inputs, outputs, reference implementation)
- **Solution** → Optimized implementations from AI agents (GPT, Claude, Gemini, etc.) or human experts
- **Workload** → Real-world shapes captured from production
- **Evaluation** → Correctness and performance measurements
- **Leaderboard & Apply** → Kernels are ranked and automatically substituted in production

![FlashInfer-Bench Pipeline](https://flashinfer.ai/assets/imgs/flashinfer-bench/image9.png)


### 1.1 Loading the FlashInfer-Trace Dataset

The [**FlashInfer-Trace**](https://huggingface.co/datasets/flashinfer-ai/flashinfer-trace) dataset contains kernel definitions, solutions (from AI agents and humans), workloads, and evaluation traces.


In [23]:
# pip install flashinfer-bench, or pip install -v .e . 
import flashinfer_bench as fib
import os
# Load the FlashInfer-Trace dataset
TRACE_PATH = os.environ.get("FLASHINFER_TRACE_PATH", "./flashinfer-trace")
trace_set = fib.TraceSet.from_path(TRACE_PATH)
# Display summary statistics
print(f"📊 FLASHINFER-TRACE DATASET Statistics:")
print(f"   • Total Definitions: {len(trace_set.definitions)}")
print(f"   • Total Solutions:   {sum(len(s) for s in trace_set.solutions.values())}")
print(f"   • Total Workloads:   {sum(len(w) for w in trace_set.workloads.values())}")
print(f"   • Total Traces:      {sum(len(t) for t in trace_set.traces.values())}")
# Show trace summary
summary = trace_set.summary()
print(f"✅ Passed evaluations: {summary['passed']}/{(summary['passed'] + summary['failed'])}")
print(f"⏱️  Avg latency: {summary['avg_latency_ms']:.4f} ms")


📊 FLASHINFER-TRACE DATASET Statistics:
   • Total Definitions: 35
   • Total Solutions:   315
   • Total Workloads:   1549
   • Total Traces:      6702
✅ Passed evaluations: 4098/6702
⏱️  Avg latency: 11.3755 ms


### 1.2 Definition

A **Definition** specifies the kernel's contract: axes (dimensions), inputs, outputs, and a reference implementation.


In [29]:
# List all available definitions by operation type
from collections import defaultdict
definitions_by_type = defaultdict(list)
for name, defn in trace_set.definitions.items():
    definitions_by_type[defn.op_type].append(name)

print("Available definitions")

for op_type, defs in sorted(definitions_by_type.items()):
    print(f"🔷 {op_type.upper()} ({len(defs)} definitions)")
    for d in sorted(defs):
        print(f"   └── {d}")

Available definitions
🔷 GEMM (8 definitions)
   └── gemm_n128_k2048
   └── gemm_n2048_k4096
   └── gemm_n256_k7168
   └── gemm_n28672_k4096
   └── gemm_n4096_k14336
   └── gemm_n4096_k4096
   └── gemm_n5120_k2048
   └── gemm_n6144_k4096
🔷 GQA_PAGED (4 definitions)
   └── gqa_paged_decode_h32_kv4_d128_ps1
   └── gqa_paged_decode_h32_kv8_d128_ps1
   └── gqa_paged_prefill_causal_h32_kv4_d128_ps1
   └── gqa_paged_prefill_causal_h32_kv8_d128_ps1
🔷 GQA_RAGGED (2 definitions)
   └── gqa_ragged_prefill_causal_h32_kv4_d128
   └── gqa_ragged_prefill_causal_h32_kv8_d128
🔷 MLA_PAGED (2 definitions)
   └── mla_paged_decode_h16_ckv512_kpe64_ps1
   └── mla_paged_prefill_causal_h16_ckv512_kpe64_ps1
🔷 MOE (1 definitions)
   └── moe_fp8_block_scale_ds_routing_topk8_ng8_kg4_e32_h7168_i2048
🔷 RMSNORM (9 definitions)
   └── fused_add_rmsnorm_h2048
   └── fused_add_rmsnorm_h4096
   └── fused_add_rmsnorm_h7168
   └── rmsnorm_h128
   └── rmsnorm_h1536
   └── rmsnorm_h2048
   └── rmsnorm_h4096
   └── rmsnorm_h

In [39]:
# Examine a specific definition: fused_add_rmsnorm_h4096 (used in Llama-3.1-8B)
defn = trace_set.definitions["fused_add_rmsnorm_h4096"]
print(f"\n DEFINITION: {def_name}")
print(defn.model_dump_json(indent=2))
# formatted reference implemention
# print(defn.reference)


 DEFINITION: fused_add_rmsnorm_h4096
{
  "name": "fused_add_rmsnorm_h4096",
  "op_type": "rmsnorm",
  "axes": {
    "batch_size": {
      "type": "var",
      "description": null
    },
    "hidden_size": {
      "type": "const",
      "value": 4096,
      "description": null
    }
  },
  "inputs": {
    "hidden_states": {
      "shape": [
        "batch_size",
        "hidden_size"
      ],
      "dtype": "bfloat16",
      "description": null
    },
    "residual": {
      "shape": [
        "batch_size",
        "hidden_size"
      ],
      "dtype": "bfloat16",
      "description": null
    },
    "weight": {
      "shape": [
        "hidden_size"
      ],
      "dtype": "bfloat16",
      "description": null
    }
  },
  "outputs": {
    "output": {
      "shape": [
        "batch_size",
        "hidden_size"
      ],
      "dtype": "bfloat16",
      "description": null
    }
  },
  "reference": "import torch\n\n@torch.no_grad()\ndef run(hidden_states, residual, weight):\n    _, hid

### 1.3 Solutions (AI Agent & Human Expert Kernels)
A **Solution** is a concrete implementation of a Definition's interface. Solutions can come from:
- **AI Agents**: GPT, Claude Opus, Gemini, etc
- **Human Experts**: Engineers writing optimized CUDA/Triton code
- **Baseline**: FlashInfer's reference implementations


In [42]:
# List solutions for our target definition
from collections import defaultdict
solutions = trace_set.solutions.get(def_name, [])

print(f"Solution For {def_name}")
print(f"Total solutions: {len(solutions)}")

# Group by author
by_author = defaultdict(list)
for sol in solutions:
    by_author[sol.author].append(sol)

for author, sols in sorted(by_author.items()):
    print(f"👤 Author: {author}")
    for sol in sols:
        lang = sol.spec.language.value if hasattr(sol.spec.language, 'value') else sol.spec.language
        target = ", ".join(sol.spec.target_hardware) if sol.spec.target_hardware else "any"
        print(f"   └── {sol.name}")
        print(f"       Language: {lang} | Target: {target}")


Solution For fused_add_rmsnorm_h4096
Total solutions: 9
👤 Author: claude-opus-4-1-20250805
   └── claude-opus-4-1_cuda_462ef5
       Language: cuda | Target: B200
   └── claude-opus-4-1_triton_f41fa3
       Language: triton | Target: B200
👤 Author: flashinfer
   └── flashinfer_wrapper_0ff432
       Language: python | Target: NVIDIA GeForce RTX 4090, NVIDIA A100, NVIDIA H20, NVIDIA H100, NVIDIA H200, NVIDIA B200
👤 Author: gemini-2.5-pro
   └── gemini-2.5-pro_cuda_5808cd
       Language: cuda | Target: B200
   └── gemini-2.5-pro_triton_dc28mj
       Language: triton | Target: B200
👤 Author: gpt-5-2025-08-07
   └── gpt-5_cuda_727b5d
       Language: cuda | Target: B200
   └── gpt-5_triton_0de5b5
       Language: triton | Target: B200
👤 Author: gpt-o3
   └── gpt-o3_cuda_a7bbcf
       Language: cuda | Target: B200
   └── gpt-o3_triton_c1e819
       Language: triton | Target: B200


In [43]:
# Look at an AI-generated solution (GPT-5 Triton)
gpt5_sol = next((s for s in solutions if "gpt-5" in s.author.lower() and "triton" in s.spec.language.value.lower()), None)

print(f"\n📜 AI Generated Kernel Code:")
for src in gpt5_sol.sources:
    print(f"# File: {src.path}")
    print(src.content)



📜 AI Generated Kernel Code:
# File: main.py
import torch
import triton
import triton.language as tl


@triton.jit
def fused_add_rmsnorm_h4096_kernel(
    hidden_ptr, residual_ptr, weight_ptr, output_ptr,
    M,  # number of rows (batch size)
    stride_hs_m, stride_hs_n,
    stride_res_m, stride_res_n,
    stride_out_m, stride_out_n,
    H: tl.constexpr,       # hidden size, must be 4096
    EPS: tl.constexpr,     # epsilon for numerical stability
    BLOCK_SIZE: tl.constexpr,
):
    tl.static_assert(H == 4096)
    pid = tl.program_id(0)
    row_in_bounds = pid < M

    cols = tl.arange(0, BLOCK_SIZE)

    # First pass: compute sum of squares across the row to get RMS
    sumsq = tl.zeros([1], dtype=tl.float32)
    for col_start in range(0, H, BLOCK_SIZE):
        off = col_start + cols
        mask = row_in_bounds & (off < H)
        hs = tl.load(hidden_ptr + pid * stride_hs_m + off * stride_hs_n, mask=mask, other=0).to(tl.float32)
        rs = tl.load(residual_ptr + pid * stride_res

## 2. Bring Your Own Kernel to FlashInfer-Bench

FlashInfer-Bench has an example **KernelGenerator** that uses LLM agents (GPT-5, Claude, Gemini, etc.) to automatically generate GPU kernels from Definition specifications.

The generator implements an **iterative optimization loop**:
1. Generate initial kernel code from the Definition
2. Evaluate correctness and performance using flashinfer-bench
3. If failed, feed error feedback back to the LLM
4. Repeat until PASSED or max rounds reached

📖 Tutorial: [bring_your_own_kernel.mdx](https://github.com/flashinfer-ai/flashinfer-bench/blob/main/docs/tutorials/bring_your_own_kernel.mdx)

In [45]:
# Initialize the KernelGenerator with LLM credentials from environment
# Before running: export LLM_API_KEY="your-api-key" BASE_URL="https://api.openai.com/v1"
import os
import sys

sys.path.insert(0, os.path.join(os.path.abspath("."), "examples/kernel_generator"))
from kernel_generator import KernelGenerator

# Check for required environment variables
api_key = os.getenv("LLM_API_KEY")
base_url = os.getenv("BASE_URL", "https://api.openai.com/v1")

if not api_key:
    raise EnvironmentError(
        "LLM_API_KEY not set. Run:\n"
        "  export LLM_API_KEY='your-api-key'\n"
        "  export BASE_URL='https://api.openai.com/v1'"
    )

# Initialize generator with your preferred model
generator = KernelGenerator(
    model_name="gpt-4o",
    language="triton",
    target_gpu="B200",
    api_key=api_key,
    base_url=base_url,
    reasoning_effort="high",
)
print(f"KernelGenerator initialized")
print(f"Model: {generator.model_name}")
print(f"Language: {generator.language}")
print(f"Target GPU: {generator.target_gpu}")


KernelGenerator initialized
Model: gpt-4o
Language: triton
Target GPU: B200


In [46]:
# Generate an optimized kernel using the LLM agent
# Pick a definition to generate a kernel for (using fused_add_rmsnorm_h4096 as an example)
target_def_name = "fused_add_rmsnorm_h4096"
target_defn = trace_set.definitions[target_def_name]

print(f"Generating Kernel for : {target_def_name}")
print(f"Kernel description: {target_defn.description}")

# Run the iterative optimization loop
solution = generator.generate(
    traceset=trace_set,
    definition=target_defn,
    max_opt_rounds=10,
)

# Display the generated solution
print("\n" + "="*60)
print(f"Generated solution source code for: {solution.name}")
print("="*60)

for src in solution.sources:
    print(f"📄 {src.path}")
    print(src.content)

Generating Kernel for : fused_add_rmsnorm_h4096
Kernel description: Fused Add + RMSNorm with hidden_size=4096 for Llama-3.1-8B. Epsilon is fixed at 1e-5.
Generating optimized solution for fused_add_rmsnorm_h4096
Using workload 7f65318b-59ac-478e-84ed-d19052b65a7e for optimization feedback

=== Optimization Round 1/10 ===
Evaluating solution...


Failed to discover resources for CUDA package 'flashinfer_bench.thirdparty.cutlass'; continuing without it.
Traceback (most recent call last):
  File "/home/yongwww/workspace/flashinfer-bench/flashinfer_bench/compile/builders/cuda_builder.py", line 32, in _get_package_paths
    include_dir = resources.files(pkg_name) / "include"
                  ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 46, in wrapper
    return func(anchor)
           ^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 56, in files
    return from_package(resolve(anchor))
                        ^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/functools.py", line 912, in wrapper
    return dispatch(args[0].__class__)(*args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 82, in _
    return im

Evaluation status: RUNTIME_ERROR
Solution failed with RUNTIME_ERROR, extracting feedback for next round...
Error details:
Traceback (most recent call last):
  File "/home/yongwww/miniconda3/lib/python3.12/site-packages/triton/language/core.py", line 42, in wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/site-packages/triton/language/core.py", line 2045, in dot
    return _semantic.dot(input, other, acc, input_precision, max_num_imprecise_acc, out_dtype)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/site-packages/triton/language/semantic.py", line 1497, in dot
    assert lhs_rank == rhs_rank == 2 or lhs_rank == rhs_rank == 3, f"Both inputs must be either 2D or 3D; (lhs: {lhs.shape} vs rhs: {rhs.shape})"
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: Both inputs must be either 2D or 3D; (l

Failed to discover resources for CUDA package 'flashinfer_bench.thirdparty.cutlass'; continuing without it.
Traceback (most recent call last):
  File "/home/yongwww/workspace/flashinfer-bench/flashinfer_bench/compile/builders/cuda_builder.py", line 32, in _get_package_paths
    include_dir = resources.files(pkg_name) / "include"
                  ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 46, in wrapper
    return func(anchor)
           ^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 56, in files
    return from_package(resolve(anchor))
                        ^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/functools.py", line 912, in wrapper
    return dispatch(args[0].__class__)(*args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 82, in _
    return im

Evaluation status: INCORRECT_NUMERICAL
Solution failed with INCORRECT_NUMERICAL, extracting feedback for next round...
Generating optimized code for round 3...

=== Optimization Round 3/10 ===
Evaluating solution...


...

Evaluation status: RUNTIME_ERROR
Solution failed with RUNTIME_ERROR, extracting feedback for next round...
Error details:
Traceback (most recent call last):
  File "/home/yongwww/workspace/flashinfer-bench/flashinfer_bench/bench/evaluators/default.py", line 113, in check_correctness
    out = sol_runnable(**inp)
          ^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/workspace/flashinfer-bench/flashinfer_bench/compile/runnable.py", line 27, in __call__
    ret = self._fn(**kwargs)
          ^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/.cache/flashinfer_bench/python/tmpnqdu41gn/fib_py_gpt_4o_fused_add_rmsnorm_h4096_triton_optimized_r8/main.py", line 73, in run
    fused_add_rmsnorm_kernel[grid](
  File "/home/yongwww/miniconda3/lib/python3.12/site-packages/triton/runtime/jit.py", line 390, in <lambda>
    return lambda *args, **kwargs: self.run(grid=grid, warmup=False, *args, **kwargs)
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/mi

Failed to discover resources for CUDA package 'flashinfer_bench.thirdparty.cutlass'; continuing without it.
Traceback (most recent call last):
  File "/home/yongwww/workspace/flashinfer-bench/flashinfer_bench/compile/builders/cuda_builder.py", line 32, in _get_package_paths
    include_dir = resources.files(pkg_name) / "include"
                  ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 46, in wrapper
    return func(anchor)
           ^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 56, in files
    return from_package(resolve(anchor))
                        ^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/functools.py", line 912, in wrapper
    return dispatch(args[0].__class__)(*args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 82, in _
    return im

Evaluation status: INCORRECT_NUMERICAL
Solution failed with INCORRECT_NUMERICAL, extracting feedback for next round...
Generating optimized code for round 10...

=== Optimization Round 10/10 ===
Evaluating solution...


Failed to discover resources for CUDA package 'flashinfer_bench.thirdparty.cutlass'; continuing without it.
Traceback (most recent call last):
  File "/home/yongwww/workspace/flashinfer-bench/flashinfer_bench/compile/builders/cuda_builder.py", line 32, in _get_package_paths
    include_dir = resources.files(pkg_name) / "include"
                  ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 46, in wrapper
    return func(anchor)
           ^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 56, in files
    return from_package(resolve(anchor))
                        ^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/functools.py", line 912, in wrapper
    return dispatch(args[0].__class__)(*args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yongwww/miniconda3/lib/python3.12/importlib/resources/_common.py", line 82, in _
    return im

Evaluation status: INCORRECT_NUMERICAL
Reached maximum rounds (10), returning current solution

Generated solution source code for: gpt-4o_fused_add_rmsnorm_h4096_triton_optimized_r10
📄 main.py
import torch
import triton
import triton.language as tl

@triton.jit
def fused_add_rmsnorm_kernel(
    hidden_states_ptr, residual_ptr, weight_ptr, output_ptr, 
    num_elements, hidden_size, eps, BLOCK_SIZE: tl.constexpr
):
    # Compute program ID for the grid
    pid = tl.program_id(0)
    
    # Compute the offsets for this block
    start = pid * BLOCK_SIZE
    offsets = start + tl.arange(0, BLOCK_SIZE)
    
    # Ensure the offsets are within bounds
    mask = offsets < num_elements
    
    # Load hidden states and residuals with masking
    hidden_states = tl.load(hidden_states_ptr + offsets, mask=mask, other=0.0)
    residual = tl.load(residual_ptr + offsets, mask=mask, other=0.0)
    
    # Compute x as the sum of hidden states and residuals
    x = hidden_states + residual
    
    # 

📖 Full tutorial: [bring_your_own_kernel.mdx](https://github.com/flashinfer-ai/flashinfer-bench/blob/main/docs/tutorials/bring_your_own_kernel.mdx)


---
## 3. The Leaderboard

The **FlashInfer-Bench Leaderboard** ([bench.flashinfer.ai](https://bench.flashinfer.ai/)) provides:
- **Global Author Ranking**: Aggregates performance metrics to rank contributors
- **Drill-Down Analysis**: Detailed insights into specific kernel definitions
- **fast_p Metric**: Fraction of workloads where a kernel runs 'p' times faster than baseline


In [49]:
# Analyze author performance across all definitions
import pandas as pd
author_stats = defaultdict(lambda: {"total": 0, "passed": 0, "total_speedup": 0, "wins": 0})

for def_name_iter, traces_list in trace_set.traces.items():
    for trace in traces_list:
        sol = trace_set.get_solution(trace.solution)
        if sol:
            author = sol.author
            author_stats[author]["total"] += 1
            
            if trace.evaluation and trace.evaluation.status.value == "PASSED":
                author_stats[author]["passed"] += 1

print("Local Leaderboard:")

# Calculate average speedup and sort
leaderboard = []
for author, stats in author_stats.items():
    if stats["passed"] > 0:
        avg_speedup = stats["total_speedup"] / stats["passed"]
        win_rate = stats["wins"] / stats["passed"] * 100
        leaderboard.append({
            "Author": author,
            "Total Traces": stats["total"],
            "Passed": stats["passed"],
        })

leaderboard_df = pd.DataFrame(leaderboard)
leaderboard_df = leaderboard_df.sort_values("Passed", ascending=False)
print(leaderboard_df.to_string(index=False))

Local Leaderboard:
                  Author  Total Traces  Passed
        gpt-5-2025-08-07          1412    1142
                  gpt-o3          1445    1019
              flashinfer          1108     631
claude-opus-4-1-20250805          1457     559
          gemini-2.5-pro           961     469
                 PyTorch           311     277
                  gpt-4o             1       1


---
## 4. End-to-End Apply with SGLang & Llama 3.1

The **Apply** feature allows you to replace kernels in FlashInfer with the best-performing ones from the trace database. This is the key to the "virtuous cycle" - improvements flow directly to production!

### How it works
When you call enable_apply(), FlashInfer-Bench automatically installs lightweight adapters that:
- Intercept FlashInfer wrapper methods (plan and run)
- Extract runtime parameters and match them to definitions
- Dispatch to the best-performing solution from your traces
- Fall back to the original FlashInfer implementation if no suitable solution exists


### 4.1 Starting SGLang serving with and without FlashInfer-Bench Apply

To enable kernel substitution, we set environment variables before launching SGLang:


In [11]:
print("="*70)
print("      Step 1: Launch SGLang serving endpoint with FlashInfer-Bench Apply")
print("="*70)
print("""
# Set environment variables to enable kernel substitution
export FIB_ENABLE_APPLY=1
export FIB_DATASET_PATH=/sgl-workspace/flashinfer-trace

# Launch SGLang server with Llama-3.1-8B-Instruct
cd /sgl-workspace/sglang
python3 -m sglang.launch_server \\
    --model-path meta-llama/Llama-3.1-8B-Instruct \\
    --cuda-graph-max-bs 8 \\
    --disable-radix-cache

# You should see "FlashInfer-Bench Apply for <kernel_name>" messages
# indicating that optimized kernels are being substituted!
""")


      Step 1: Launch SGLang serving endpoint with FlashInfer-Bench Apply

# Set environment variables to enable kernel substitution
export FIB_ENABLE_APPLY=1
export FIB_DATASET_PATH=/sgl-workspace/flashinfer-trace

# Launch SGLang server with Llama-3.1-8B-Instruct
cd /sgl-workspace/sglang
python3 -m sglang.launch_server \
    --model-path meta-llama/Llama-3.1-8B-Instruct \
    --cuda-graph-max-bs 8 \
    --disable-radix-cache

# You should see "FlashInfer-Bench Apply for <kernel_name>" messages
# indicating that optimized kernels are being substituted!



### 4.2 Benchmarking SGLang serving

Once the server is running, use `sglang.bench_serving` to measure performance:


In [12]:
print("="*70)
print("      Step 2: Collect the Serving perf numbers")
print("="*70)
print("""
# Set target model and batch size
tgt_model="meta-llama/Llama-3.1-8B-Instruct"
bs=8  # Concurrency

# Run the benchmark
python3 -m sglang.bench_serving \\
    --backend sglang \\
    --model ${tgt_model} \\
    --num-prompts $((50 * bs)) \\
    --sharegpt-output-len 100 \\
    --max-concurrency $bs

# Example with different batch sizes:
for bs in 1 2 4 8; do
    echo "=== Benchmarking with concurrency=$bs ==="
    python3 -m sglang.bench_serving \\
        --backend sglang \\
        --model meta-llama/Llama-3.1-8B-Instruct \\
        --num-prompts $((50 * bs)) \\
        --sharegpt-output-len 100 \\
        --max-concurrency $bs
done
""")


      Step 2: Collect the Serving perf numbers

# Set target model and batch size
tgt_model="meta-llama/Llama-3.1-8B-Instruct"
bs=8  # Concurrency

# Run the benchmark
python3 -m sglang.bench_serving \
    --backend sglang \
    --model ${tgt_model} \
    --num-prompts $((50 * bs)) \
    --sharegpt-output-len 100 \
    --max-concurrency $bs

# Example with different batch sizes:
for bs in 1 2 4 8; do
    echo "=== Benchmarking with concurrency=$bs ==="
    python3 -m sglang.bench_serving \
        --backend sglang \
        --model meta-llama/Llama-3.1-8B-Instruct \
        --num-prompts $((50 * bs)) \
        --sharegpt-output-len 100 \
        --max-concurrency $bs
done



---
## 5. Summary

### Key Takeaways:

1. **FlashInfer-Bench** creates a virtuous cycle where AI agents and human experts collaborate to optimize GPU kernels.

2. **The Pipeline**:
   - **Definitions**: Formal specifications of kernel interfaces
   - **Solutions**: Implementations from AI (GPT-5, Claude, Gemini) and humans
   - **Workloads**: Real-world input shapes captured from production
   - **Evaluations**: Correctness and performance measurements

3. **The Leaderboard** ([bench.flashinfer.ai](https://bench.flashinfer.ai/)) ranks contributors and tracks the best kernels.

4. **End-to-End Apply with SGLang**:
   ```bash
   # Enable kernel substitution
   export FIB_ENABLE_APPLY=1
   export FIB_DATASET_PATH=/sgl-workspace/flashinfer-trace
   
   # Launch server
   python3 -m sglang.launch_server --model-path meta-llama/Llama-3.1-8B-Instruct ...
   
   # Benchmark
   python3 -m sglang.bench_serving --backend sglang --model ... --num-prompts ...
   ```

5. **Works with SGLang, vLLM, and other LLM frameworks** - improvements flow directly to production!
